# Logistic Regression: Multiclass Strategies

Comparative analysis of the different multiclass classification strategies
available in scikit-learn's `LogisticRegression`:

- **Multinomial (native softmax)**: a single coefficient matrix, probabilities
  sum to 1 via the softmax function. Assumes that the classes are mutually exclusive.
- **One-vs-Rest (OvR)**: one binary classifier per class (each class vs.
  all the others). Probabilities are calibrated independently.
- **One-vs-One (OvO)**: one binary classifier for each pair of classes.
  Final prediction by voting among the pairs.

Dataset: `jp797498e/twitter-entity-sentiment-analysis` (4 classes: Positive,
Negative, Neutral, Irrelevant).

In [1]:
# ─── Imports ────────────────────────────────────────────────────
import kagglehub
import pandas as pd
import numpy as np
import re
import time
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsOneClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score
)

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

SEED = 42
np.random.seed(SEED)

print('Imports OK \u2714')

Imports OK ✔


## 1. Loading and Preprocessing

In [2]:
# ─── Loads data via kagglehub ───────────────────────────────────────────
path = kagglehub.dataset_download('jp797498e/twitter-entity-sentiment-analysis')
train_df = pd.read_csv(os.path.join(path, 'twitter_training.csv'))
val_df = pd.read_csv(os.path.join(path, 'twitter_validation.csv'))

print(f'Train : {train_df.shape}')
print(f'Valida\u00e7\u00e3o: {val_df.shape}')

Treino : (74681, 4)
Validação: (999, 4)


In [3]:
# ─── Standardizes column names ──────────────────────────────────────────────
COL_NAMES = ['id', 'entity', 'sentiment', 'text']

if train_df.shape[1] == 4:
    train_df.columns = COL_NAMES
    val_df.columns   = COL_NAMES

print('Unique sentiments:', train_df['sentiment'].unique())
print('Training distribution:')
print(train_df['sentiment'].value_counts())

Sentimentos únicos: ['Positive' 'Negative' 'Neutral' 'Irrelevant']
Distribuição no treino:
sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [4]:
# ─── Cleaning function ───────────────────────────────────────────────
def clean_tweet(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^a-z0-9\s!?.,\'\-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for df in (train_df, val_df):
    df['clean_text'] = df['text'].apply(clean_tweet)

valid_sentiments = ['Positive', 'Negative', 'Neutral', 'Irrelevant']
train_df = train_df[
    train_df['clean_text'].str.len() > 0 &
    train_df['sentiment'].isin(valid_sentiments)
].copy()

val_df = val_df[
    val_df['clean_text'].str.len() > 0 &
    val_df['sentiment'].isin(valid_sentiments)
].copy()

print(f'Train after cleaning : {train_df.shape}')
print(f'Validation after cleaning: {val_df.shape}')

Treino após limpeza : (73768, 6)
Validação após limpeza: (999, 6)


In [5]:
# ─── Encode labels ─────────────────────────────────────────────────
le = LabelEncoder()
le.fit(valid_sentiments)

y_train = le.transform(train_df['sentiment'])
y_val   = le.transform(val_df['sentiment'])

X_train_text = train_df['clean_text'].values
X_val_text   = val_df['clean_text'].values

print('Classes:', dict(zip(le.classes_, le.transform(le.classes_))))

Classes: {'Irrelevant': 0, 'Negative': 1, 'Neutral': 2, 'Positive': 3}


## 2. TF-IDF Vectorization

In [6]:
# ─── TF-IDF Vectorizer ──────────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features=70_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    strip_accents='unicode',
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf   = tfidf.transform(X_val_text)

print(f'TF-IDF train shape : {X_train_tfidf.shape}')
print(f'Shape TF-IDF valida\u00e7\u00e3o: {X_val_tfidf.shape}')

Shape TF-IDF treino : (73768, 70000)
Shape TF-IDF validação: (999, 70000)


## 3. Multi-Class Experiments

We test 5 configurations that cover all the multiclass strategies
available in scikit-learn:

| # | Strategy | `multi_class` | `solver` | Description |
|---|--------------|---------------|----------|-------------------|
| 1 | Multinomial  | `multinomial` | `lbfgs`   | Native softmax (one weight matrix W \u00d7 K) |
| 2 | OvR (lbfgs)  | `ovr`         | `lbfgs`   | One-vs-Rest with a quasi-Newton solver |
| 3 | OvR (liblinear) | `ovr`     | `liblinear` | One-vs-Rest with coordinate descent |
| 4 | OvR (saga)   | `ovr`         | `saga`    | One-vs-Rest with stochastic gradient |
| 5 | OvO (liblinear) | (wrap)     | `liblinear` | One-vs-One via `OneVsOneClassifier` |

Each configuration is evaluated on 4 values of C (inverse regularization).

In [7]:
# ─── Experiment settings ───────────────────────────────────────────────
C_VALUES = [0.1, 1.0, 10.0, 100.0]

STRATEGIES = [
    ('Multinomial (lbfgs)', 'multinomial', 'lbfgs', False),
    ('OvR (lbfgs)',         'ovr',         'lbfgs', False),
    ('OvR (liblinear)',     'ovr',         'liblinear', False),
    ('OvR (saga)',          'ovr',         'saga', False),
    ('OvO (liblinear)',     'ovr',         'liblinear', True),
]

results_rows = []

header = f'{"Strategy":<22s} {"C":<6s} {"Accuracy":<10s} {"F1-w":<10s} {"Time(s)":<10s}'
print(header)
print('-' * 58)

for strat_name, multi_class, solver, is_ovo in STRATEGIES:
    for C in C_VALUES:
        if is_ovo:
            base_lr = LogisticRegression(
                C=C, max_iter=2000, solver=solver,
                multi_class=multi_class, random_state=SEED
            )
            model = OneVsOneClassifier(base_lr, n_jobs=-1)
        else:
            model = LogisticRegression(
                C=C, max_iter=2000, solver=solver,
                multi_class=multi_class, random_state=SEED,
                n_jobs=-1 if solver != 'liblinear' else 1
            )

        t0 = time.time()
        model.fit(X_train_tfidf, y_train)
        t_train = time.time() - t0

        preds = model.predict(X_val_tfidf)
        acc = accuracy_score(y_val, preds)
        f1 = f1_score(y_val, preds, average='weighted')

        results_rows.append({
            'strategy': strat_name,
            'C': C,
            'accuracy': acc,
            'f1_weighted': f1,
            'train_time': t_train,
            'model': model,
        })

        print(f'{strat_name:<22s} {C:<6.1f} {acc:<10.4f} {f1:<10.4f} {t_train:<10.2f}')

results_df = pd.DataFrame(results_rows)
print('\nExperiments complete \u2714')

Estrategia             C      Acuracia   F1-w       Tempo(s)  
----------------------------------------------------------
Multinomial (lbfgs)    0.1    0.7598     0.7516     5.32      
Multinomial (lbfgs)    1.0    0.9750     0.9750     13.99     
Multinomial (lbfgs)    10.0   0.9820     0.9820     59.93     
Multinomial (lbfgs)    100.0  0.9780     0.9780     73.18     
OvR (lbfgs)            0.1    0.7137     0.6980     5.40      
OvR (lbfgs)            1.0    0.9630     0.9630     10.12     
OvR (lbfgs)            10.0   0.9780     0.9780     12.86     
OvR (lbfgs)            100.0  0.9800     0.9800     37.14     
OvR (liblinear)        0.1    0.7137     0.6980     3.49      
OvR (liblinear)        1.0    0.9630     0.9630     6.34      
OvR (liblinear)        10.0   0.9780     0.9780     13.21     
OvR (liblinear)        100.0  0.9790     0.9790     43.11     
OvR (saga)             0.1    0.7137     0.6980     3.77      
OvR (saga)             1.0    0.9630     0.9630     3.77   

## 4. Per-Class Analysis (C=10.0)

In [8]:
# ─── Detailed evaluation with C=10.0 ──────────────────────────────────
BEST_C = 10.0

detailed = []

for strat_name, multi_class, solver, is_ovo in STRATEGIES:
    if is_ovo:
        base_lr = LogisticRegression(
            C=BEST_C, max_iter=2000, solver=solver,
            multi_class=multi_class, random_state=SEED
        )
        model = OneVsOneClassifier(base_lr, n_jobs=-1)
    else:
        model = LogisticRegression(
            C=BEST_C, max_iter=2000, solver=solver,
            multi_class=multi_class, random_state=SEED,
            n_jobs=-1 if solver != 'liblinear' else 1
        )

    t0 = time.time()
    model.fit(X_train_tfidf, y_train)
    t_train = time.time() - t0

    preds = model.predict(X_val_tfidf)
    report = classification_report(y_val, preds,
                                   target_names=le.classes_,
                                   output_dict=True)

    detailed.append({
        'strategy': strat_name,
        'model': model,
        'preds': preds,
        'report': report,
        'train_time': t_train,
    })

    print(f'\n{"="*60}')
    print(f'  {strat_name} (C={BEST_C})')
    print('='*60)
    print(f'Training: {t_train:.2f}s')
    print(classification_report(y_val, preds, target_names=le.classes_))

print('Detailed evaluation complete \u2714')


  Multinomial (lbfgs) (C=10.0)
Treino: 135.43s
              precision    recall  f1-score   support

  Irrelevant       0.99      0.98      0.99       171
    Negative       0.99      0.98      0.99       266
     Neutral       0.99      0.97      0.98       285
    Positive       0.96      0.99      0.98       277

    accuracy                           0.98       999
   macro avg       0.98      0.98      0.98       999
weighted avg       0.98      0.98      0.98       999


  OvR (lbfgs) (C=10.0)
Treino: 22.39s
              precision    recall  f1-score   support

  Irrelevant       0.99      0.98      0.98       171
    Negative       0.98      0.98      0.98       266
     Neutral       0.99      0.97      0.98       285
    Positive       0.96      0.98      0.97       277

    accuracy                           0.98       999
   macro avg       0.98      0.98      0.98       999
weighted avg       0.98      0.98      0.98       999


  OvR (liblinear) (C=10.0)
Treino: 15.72s


## 5. Final Comparative Table (C=10.0)

In [9]:
# ─── Summary table ──────────────────────────────────────────────────
summary_rows = []
for entry in detailed:
    r = entry['report']
    row = {
        'Strategy': entry['strategy'],
        'Time (s)': f"{entry['train_time']:.2f}",
        'Accuracy': f"{r['accuracy']:.4f}",
        'F1 (weighted)': f"{r['weighted avg']['f1-score']:.4f}",
        'F1 (macro)': f"{r['macro avg']['f1-score']:.4f}",
    }
    for cls_name in le.classes_:
        row[f'F1 {cls_name}'] = f"{r[cls_name]['f1-score']:.4f}"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df.style
    .set_caption('Comparison of the Multiclass Strategies (C=10.0)')
    .background_gradient(cmap='Blues', subset=[c for c in summary_df.columns
                                                if c not in ['Strategy', 'Time (s)']])
)

Estratégia,Tempo (s),Acurácia,F1 (weighted),F1 (macro),F1 Irrelevant,F1 Negative,F1 Neutral,F1 Positive
Multinomial (lbfgs),135.43,0.9820,0.9820,0.9829,0.9853,0.9857,0.9798,0.9767
OvR (lbfgs),22.39,0.9780,0.9779,0.9777,0.9823,0.9809,0.9712,0.9635
OvR (liblinear),15.72,0.9780,0.9779,0.9777,0.9823,0.9809,0.9712,0.9635
OvR (saga),10.07,0.9780,0.9779,0.9777,0.9823,0.9809,0.9712,0.9635
OvO (liblinear),3.89,0.9770,0.9770,0.9768,0.9758,0.9810,0.9744,0.9738


## 6. Conclusions

### Main findings

| Strategy | Best C | Accuracy | F1-weighted | Time (s) |
|------------|----------|-----------|-------------|----------|
| **Multinomial (lbfgs)** | 10 | **0,9820** | **0,9820** | 59,93 |
| OvR (lbfgs) | 100 | 0,9800 | 0,9800 | 37,14 |
| OvR (liblinear) | 100 | 0,9790 | 0,9790 | 43,11 |
| OvR (saga) | 100 | 0,9790 | 0,9790 | 41,66 |
| OvO (liblinear) | 100 | 0,9780 | 0,9780 | 6,82 |

### Analysis

1. **Multinomial (softmax) is the most accurate, but drastically slower.**
   The accuracy of 0,9820 exceeds the other strategies by up to 0,4 pp, but the
   training time with C=10 is 59,93s — about **5× slower** than OvR and
   **28× slower** than OvO. This happens because the multinomial method solves
   a joint problem of K classes simultaneously, while OvR and OvO
   break the problem into smaller, parallelizable subproblems.

2. **OvR with C=100 technically ties Multinomial C=10.**
   OvR (lbfgs) with C=100 reaches 0,9800 vs 0,9820 for Multinomial C=10. The
   difference is marginal (0,2 pp) and OvR is much faster (37,14s vs
   59,93s). For practical purposes, they are equivalent.

3. **OvO is the fastest, but the least accurate.**
   The One-vs-One strategy with liblinear trains K×(K-1)/2 = 6 binary
   classifiers for 4 classes. Each subproblem is small (only 2 classes), which
   speeds up training (2,16s for C=10). However, voting between pairs can
   lose information that the global view of softmax captures.

4. **The saga solver is the most efficient among the OvR variants.**
   With C=10, saga trains in 9,36s vs 12,86s for lbfgs and 13,21s for liblinear,
   keeping the same accuracy. The stochastic gradient of saga converges faster
   on large datasets such as this one (73k samples).

5. **Strong regularization (low C) penalizes OvR/OvO more than Multinomial.**
   With C=0,1, Multinomial obtains 0,7598 while OvR drops to 0,7137 and OvO
   to 0,6907. Softmax, because it shares parameters between classes, is more
   robust to strong regularization.

### Recommendation

- **Maximum accuracy (no concern about time)**: `multi_class='multinomial',
  solver='lbfgs', C=10`
- **Best cost-benefit**: `multi_class='ovr', solver='saga', C=100`
  (accuracy 0,979 in 41,66s, with L1 and L2 support)
- **Minimum time**: `OneVsOneClassifier(LogisticRegression(solver='liblinear',
  C=100))` (0,978 in 6,82s)